# Assignment 2: fresh comparison

This is a new study. No old results are restored or included.
Run the cells in order with Internet enabled.
The default model is NVIDIA Nemotron through its hosted API, using CPU and secret NVIDIA_API_KEY.

Change CLIENT to "gemini" for Gemini (CPU, secret GEMINI_API_KEY_1), or "vllm" for Qwen (GPU T4 x2).
Each model uses naive, custom and stall_reflection, the same three tasks and seeds 42-46:
45 episodes per model, 135 across the study.
Settings: 50 environment actions per episode, temperature 0, 8,192 output tokens per call,
16 text-history messages, no images, one worker. Nemotron built-in thinking is disabled for all agents; custom planning and reflection still run.
The action cap and model-specific reasoning settings must be stated in the report.
These are bounded experiments, not full-paper benchmark reproductions.

The code comes from branch siddique/assignment-fresh. Results go only to assignment_results_fresh.
Download assignment-results-fresh.zip after each model and keep all three archives.
API failures stop the study; they are not game scores. Free quotas may require several sessions.
The final report will use only these fresh results.


In [ ]:
from pathlib import Path
import os
import subprocess

CLIENT = "nvidia"
MODEL = {
    "nvidia": "nvidia/nemotron-3.5-lightning-30b-a3b",
    "gemini": "gemini-3.5-flash-lite",
    "vllm": "Qwen/Qwen2.5-7B-Instruct",
}[CLIENT]

if CLIENT == "vllm":
    gpus = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True
    ).strip().splitlines()
    if len(gpus) != 2:
        raise RuntimeError("Select GPU T4 x2 for Qwen.")
    print("GPUs:", *gpus, sep="\n- ")
else:
    from kaggle_secrets import UserSecretsClient
    secret_name, environment_name = {
        "nvidia": ("NVIDIA_API_KEY", "NVIDIA_API_KEY"),
        "gemini": ("GEMINI_API_KEY_1", "GEMINI_API_KEY"),
    }[CLIENT]
    os.environ[environment_name] = UserSecretsClient().get_secret(secret_name)
    os.environ.pop("GOOGLE_API_KEY", None)
    os.environ.pop("GEMINI_API_KEYS", None)

TEMP_ROOT = Path("/kaggle/temp")
TEMP_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT = TEMP_ROOT / "assignment-2-fresh"
if not PROJECT.exists():
    subprocess.run([
        "git", "clone", "--branch", "siddique/assignment-fresh", "--single-branch",
        "https://github.com/siddiqueMangi2003/agentic-llm-assignment-2.git", str(PROJECT),
    ], check=True)
else:
    subprocess.run(["git", "pull", "--ff-only"], cwd=PROJECT, check=True)
os.environ["HF_HOME"] = str(TEMP_ROOT / "huggingface")
RESULTS = Path("/kaggle/working/assignment_results_fresh")
print("Model:", MODEL)
print("Code:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=PROJECT, text=True).strip())


In [ ]:
import sys

ENV = TEMP_ROOT / "balrog-fresh-py310"
PYTHON = ENV / "bin" / "python"
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.10", str(ENV)], check=True)
packages = ["-e", str(PROJECT), "setuptools<81"]
if CLIENT == "vllm":
    packages.append("vllm==0.31.0")
subprocess.run(
    [sys.executable, "-m", "uv", "pip", "install", "--python", str(PYTHON), *packages],
    check=True,
)
os.environ["PATH"] = str(ENV / "bin") + os.pathsep + os.environ["PATH"]


In [ ]:
import shutil
import time
from urllib.error import URLError
from urllib.request import urlopen

command = [
    str(PYTHON), "scripts/run_assignment.py",
    "--model", MODEL, "--client", CLIENT, "--output", str(RESULTS),
]
server = None
try:
    if CLIENT == "vllm":
        log_path = Path("/kaggle/working/vllm-fresh.log")
        with log_path.open("w") as log:
            server = subprocess.Popen([
                str(ENV / "bin" / "vllm"), "serve", MODEL,
                "--host", "127.0.0.1", "--port", "18080",
                "--tensor-parallel-size", "2", "--max-model-len", "32768",
                "--gpu-memory-utilization", "0.85",
            ], stdout=log, stderr=subprocess.STDOUT)
        for _ in range(240):
            if server.poll() is not None:
                raise RuntimeError("vLLM stopped:\n" + log_path.read_text()[-4000:])
            try:
                with urlopen("http://127.0.0.1:18080/v1/models", timeout=2):
                    break
            except (URLError, TimeoutError):
                time.sleep(5)
        else:
            raise TimeoutError("vLLM did not start:\n" + log_path.read_text()[-4000:])
    subprocess.run(command, cwd=PROJECT, check=True)
finally:
    if server is not None:
        server.terminate()
        try:
            server.wait(timeout=15)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait()
    if RESULTS.exists():
        archive = shutil.make_archive(
            "/kaggle/working/assignment-results-fresh", "zip",
            root_dir=RESULTS.parent, base_dir=RESULTS.name,
        )
        print("Download results and diagnostics:", archive)
